In [1]:
#| default_exp utils

# 02 · Data plumbing and checkpoints

> Image preprocessing, column normalisation, and saving checkpoints. Exports to `lewm/utils.py`.

Three small utilities that `train.py` needs. Less glamorous than the model, but
two of them encode a real lesson about multiprocessing and about scale.

In [2]:
#| export
from collections.abc import Callable
from typing import Any

import numpy as np
import torch
from lightning.pytorch import LightningModule, Trainer
from lightning.pytorch.callbacks import Callback
from omegaconf import DictConfig
from stable_pretraining import data as dt

13:42:27 | INFO  | __init__.py | JAX version 0.6.2 available.


13:42:31 | INFO  | atomic_chec~| [atomic_save] installed crash-safe checkpoint plugin (write to sibling .tmp + fsync + atomic rename)


## Image preprocessing

Two steps, in this order:

1. **`ToImage(**imagenet_stats)`** — converts `uint8` in `[0, 255]` to `float`
   and normalises by ImageNet's channel means and standard deviations. Those
   exact numbers are used because the ViT architecture is the standard one; the
   convention is kept even though the encoder here trains from scratch
   (`pretrained: false` in the config).
2. **`Resize(img_size)`** — to 224×224, what the ViT expects.

`source` and `target` name which dict key to read and write, so the same
transform can be pointed at `pixels` or at `goal`.

In [3]:
#| export
def get_img_preprocessor(
    source: str, target: str, img_size: int = 224
) -> Callable[[dict[str, Any]], dict[str, Any]]:
    """Build the pixel transform: uint8 image -> normalised, resized float tensor.

    source / target: the dict keys to read from and write to, e.g. "pixels".
    """
    imagenet_stats = dt.dataset_stats.ImageNet
    to_image = dt.transforms.ToImage(**imagenet_stats, source=source, target=target)
    resize = dt.transforms.Resize(img_size, source=source, target=target)
    return dt.transforms.Compose(to_image, resize)

## Z-score normalisation, and why it is a class

Actions, proprioception and state are raw physical numbers — PushT positions
run to several hundred pixels. Feeding those straight into a network is asking
for trouble, so each column is standardised to zero mean and unit variance:

$$z = \frac{x - \mu}{\sigma}$$

**The interesting part is why this is a class and not a closure.** The obvious
implementation is:

```python
def get_normalizer(
    mean: torch.Tensor, std: torch.Tensor
) -> Callable[[torch.Tensor], torch.Tensor]:
    return lambda x: (x - mean) / std      # <-- breaks
```

DataLoader workers are separate processes. Arguments must be **pickled** to
reach them, and Python cannot pickle a lambda or a locally-defined function. A
class instance pickles fine: Python stores the class path plus `__dict__`.

So the rule this file encodes: **anything handed to a DataLoader worker must be
picklable.** The docstring's mention of spawned workers is the symptom; this is
the cause.

In [4]:
#| export
class ZScoreNormalizer:
    """Picklable z-score normalizer — uses a class instead of a closure so it
    survives pickle when DataLoader workers are spawned (required by LanceDataset).

    A lambda or nested function would raise PicklingError when the DataLoader
    tries to send it to a worker process.
    """

    def __init__(self, mean: torch.Tensor, std: torch.Tensor) -> None:
        self.mean = mean
        self.std = std

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        return ((x - self.mean) / self.std).float()

### Fitting the statistics

`mean` and `std` are computed once over the whole column, before training.

Two details that matter:

- **`data[~torch.isnan(data).any(dim=1)]`** drops rows containing NaN. Episode
  boundaries have no valid next action, so they are stored as NaN; including
  them would poison the mean and std into NaN, and then every embedding.
- **`.clone()`** detaches the statistics from the (possibly huge) source
  tensor, so it can be garbage collected instead of being kept alive by a view.

In [5]:
#| export
def get_column_normalizer(
    dataset: Any, source: str, target: str
) -> Callable[[dict[str, Any]], dict[str, Any]]:
    """Get normalizer for a specific column in the dataset.

    Computes mean and std over the entire column once, ignoring NaN rows,
    and returns a picklable transform that applies them.
    """
    col_data = dataset.get_col_data(source)
    data = torch.from_numpy(np.array(col_data))

    # Drop NaN rows (episode boundaries) -- one NaN would make mean/std NaN.
    data = data[~torch.isnan(data).any(dim=1)]

    # .clone() so the stats don't keep the whole column tensor alive.
    mean = data.mean(0, keepdim=True).clone()
    std = data.std(0, keepdim=True).clone()
    return dt.transforms.WrapTorchTransform(ZScoreNormalizer(mean, std), source=source, target=target)

### Check: does it pickle, and does it normalise?

The claim above is testable, so test it rather than trusting the comment.

In [6]:
import pickle

norm = ZScoreNormalizer(mean=torch.tensor([10.0]), std=torch.tensor([2.0]))
x = torch.tensor([[8.0], [10.0], [12.0]])
print("normalised:", norm(x).flatten().tolist(), " (expect -1, 0, 1)")

# The class pickles...
print("class pickles:", pickle.loads(pickle.dumps(norm))(x).flatten().tolist())

# ...the equivalent lambda does not.
bad = lambda v: (v - 10.0) / 2.0
try:
    pickle.dumps(bad)
    print("lambda pickled (unexpected)")
except (pickle.PicklingError, AttributeError) as e:
    print(f"lambda fails to pickle: {type(e).__name__}")

normalised: [-1.0, 0.0, 1.0]  (expect -1, 0, 1)
class pickles: [-1.0, 0.0, 1.0]
lambda fails to pickle: PicklingError


## Saving checkpoints

A Lightning `Callback` hooks into the training loop. This one writes a
checkpoint at the end of an epoch.

Three things worth noticing:

- **`trainer.is_global_zero`** — with multiple GPUs, every process runs this
  callback. Without the guard, they would all write the same file at once.
  Only rank 0 saves.
- **The second `if`** looks redundant with the first, and usually is. It covers
  the case where `max_epochs` is not a multiple of `epoch_interval`: the final
  epoch gets saved regardless. (Both conditions true means the same epoch is
  saved twice, which is harmless.)
- **The import inside `_save`** is deliberate: it defers a
  `stable_worldmodel` import that would otherwise be paid at module import
  time, and sidesteps a circular import.

In [7]:
#| export
class SaveCkptCallback(Callback):
    """Callback to save model checkpoint after each epoch using save_pretrained.

    Args:
        run_name: checkpoint name, relative to $STABLEWM_HOME.
        cfg: the model config, stored alongside the weights so the checkpoint
            can be rebuilt without the original Hydra run.
        epoch_interval: save every N epochs.
    """

    def __init__(
        self, run_name: str, cfg: DictConfig, epoch_interval: int = 1
    ) -> None:
        super().__init__()
        self.run_name = run_name
        self.cfg = cfg
        self.epoch_interval = epoch_interval

    def on_train_epoch_end(
        self, trainer: Trainer, pl_module: LightningModule
    ) -> None:
        super().on_train_epoch_end(trainer, pl_module)

        # Multi-GPU: every rank runs this callback, but only rank 0 writes.
        if trainer.is_global_zero:
            if (trainer.current_epoch + 1) % self.epoch_interval == 0:
                self._save(pl_module.model, trainer.current_epoch + 1)

            # Always save the final epoch, even if it isn't on the interval.
            if (trainer.current_epoch + 1) == trainer.max_epochs:
                self._save(pl_module.model, trainer.current_epoch + 1)

    def _save(self, model: torch.nn.Module, epoch: int) -> None:
        # Imported here rather than at module level: defers the cost and
        # avoids a circular import.
        from stable_worldmodel.wm.utils import save_pretrained
        save_pretrained(
            model,
            run_name=self.run_name,
            config=self.cfg,
            filename=f'weights_epoch_{epoch}.pt',
        )

## Export

After editing this notebook, regenerate `lewm/utils.py` from the repository root:

```bash
uv run nbdev-export
```

In [8]:
# Export explicitly from the repository root: uv run nbdev-export